# 🔴 Solution: Paged Attention

vLLM-style paged KV cache — block table maps logical blocks to physical pages

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def paged_attention(Q, k_pages, v_pages, block_table, context_len, block_size):
    """
    分页注意力（vLLM 风格）。

    核心思想：KV 缓存不再连续存储，而是存在固定大小的页 (block) 中。
    block_table 将逻辑块索引映射到物理页索引，实现非连续内存分配。
    这样多个请求可以共享物理内存，避免碎片化。

    参数:
        Q: 查询张量, shape = (B, 1, D)  —— 单步解码（生成一个 token）
        k_pages: 键页, shape = (num_pages, block_size, D)
        v_pages: 值页, shape = (num_pages, block_size, D)
        block_table: 整数张量 (B, max_blocks)，逻辑块 → 物理页的映射
        context_len: 每条序列有效 KV token 数
        block_size: 每页 token 数

    返回: (B, 1, D) 的输出张量
    """
    B, _, D = Q.shape
    outputs = []

    # ---- 逐 batch 处理 ----
    # 每条序列可能有不同的 block_table 映射
    for b in range(B):
        # ---- Step 1: 用 block_table gather 物理页 ----
        # block_table[b] 是第 b 条序列的逻辑→物理映射，shape: (max_blocks,)
        # k_pages[block_table[b]] 用高级索引取出对应的物理页
        # shape: (max_blocks, block_size, D)
        # .reshape(-1, D) 把所有页拼接成一个长序列: (max_blocks * block_size, D)
        phys_blocks = block_table[b]
        K_gathered = k_pages[phys_blocks].reshape(-1, D)
        V_gathered = v_pages[phys_blocks].reshape(-1, D)

        # ---- Step 2: 截取到有效长度 ----
        # context_len 是实际的 KV 长度，可能小于 max_blocks * block_size
        # [:context_len] 切掉多余的 padding
        # .unsqueeze(0) 添加一个 head 维度: (1, context_len, D)
        K_ctx = K_gathered[:context_len].unsqueeze(0)  # (1, ctx, D)
        V_ctx = V_gathered[:context_len].unsqueeze(0)  # (1, ctx, D)

        # ---- Step 3: 标准缩放点积注意力 ----
        # Q[b:b+1]: (1, 1, D) 保持 3D
        # scores: (1, 1, D) @ (1, D, ctx) = (1, 1, ctx)
        scale = D ** -0.5
        scores = (Q[b:b+1] @ K_ctx.transpose(-2, -1)) * scale
        attn = torch.softmax(scores, dim=-1)  # (1, 1, ctx)
        out = attn @ V_ctx  # (1, 1, ctx) @ (1, ctx, D) = (1, 1, D)
        outputs.append(out)

    # ---- Step 4: 拼接所有 batch ----
    return torch.cat(outputs, dim=0)  # (B, 1, D)

In [ ]:
# Verify: paged output matches standard attention with contiguous layout
torch.manual_seed(0)
B, S, D = 2, 8, 16
block_size = 4
num_blocks = S // block_size

# Build contiguous K, V
K_full = torch.randn(B, S, D)
V_full = torch.randn(B, S, D)
Q = torch.randn(B, 1, D)

# Standard attention reference
scale = D ** -0.5
scores_ref = (Q @ K_full.transpose(-2, -1)) * scale
ref_out = torch.softmax(scores_ref, dim=-1) @ V_full

# Build paged structures: identity block table (pages in order)
total_pages = B * num_blocks
k_pages = torch.zeros(total_pages, block_size, D)
v_pages = torch.zeros(total_pages, block_size, D)
block_table = []
for b in range(B):
    page_ids = []
    for blk in range(num_blocks):
        pid = b * num_blocks + blk
        k_pages[pid] = K_full[b, blk*block_size:(blk+1)*block_size]
        v_pages[pid] = V_full[b, blk*block_size:(blk+1)*block_size]
        page_ids.append(pid)
    block_table.append(page_ids)

paged_out = paged_attention(Q, k_pages, v_pages, block_table, context_len=S, block_size=block_size)

print('Shape:', paged_out.shape)
print('Max diff vs reference:', (paged_out - ref_out).abs().max().item())
print('Match:', torch.allclose(paged_out, ref_out, atol=1e-5))

In [ ]:
from torch_judge import check
check('paged_attention')

## 📝 核心思路总结

1. **分页思想来自 OS**：KV 缓存按固定大小的「页」存储，block_table 做逻辑→物理映射，避免连续内存分配的碎片问题。
2. **Gather + 截取**：先用 block_table 高级索引取出物理页，reshape 拼接，再按 context_len 截取有效部分。
3. **逐 batch 处理**：每条序列的 block_table 不同，需要循环处理；实际 vLLM 会用更高效的批量 gather kernel。
4. **注意力计算不变**：分页只影响 KV 的存储和读取方式，注意力的计算逻辑（缩放点积 + softmax + 加权求和）与标准注意力完全一致。